# 06 — Linearity Analysis

Does coupling efficiency depend on the vibration **amplitude**? Dedicated
mono-frequency test signal: 8 frequencies (5–400 Hz), each a 3.5 s segment
with a 0→1 linear amplitude ramp followed by a 0.5 s fade.

Per segment (`analysis.linearity_qc`): bandpass ± 5 Hz → integrate (÷ iω) →
cable elongation δxₗ (arc-length) and reference δL (chord-projected shaker) /
Δu_ends (endpoints) → η(t); the **Hilbert envelope** |H(δL)|(t) is the
instantaneous drive amplitude the η values are plotted against.

| § | Content |
|---|---|
| 1 | Loading & geometry |
| 2 | Raw signal overview |
| 3 | Run the segment analysis |
| 4 | η(t) per frequency + η vs amplitude |
| 5 | Frequency-domain cross-check per segment |
| 6 | Extension vs compression per amplitude bin |
| 7 | Interactive dashboards (QC + slip) |
| 8 | Export → `results/linearity_eta.npz` |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Editable reload while iterating on the package
%load_ext autoreload
%autoreload 2

import ldv_analysis as la
from ldv_analysis import config, io, analysis, plotting, export

plt.rcParams.update({
    "font.size": 11, "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "figure.dpi": 120, "savefig.dpi": 300, "figure.autolayout": True,
    "axes.titlesize": 11, "axes.labelsize": 10, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
    "lines.linewidth": 1.2,
})
print(f"{len(la.ALL_DATASETS)} datasets in catalogue.")

---
## § 1  Loading & geometry

In [ ]:
ACTIVE_LABELS = config.LINEARITY_LABELS
DATASETS = config.select_datasets(ACTIVE_LABELS)

for cfg in DATASETS:
    io.load_cable_dataset(cfg)
    analysis.prepare_geometry(cfg)
    print(f"  {cfg['label']:30s} L={cfg['chord_len']*100:.2f} cm  "
          f"sag={cfg['static_sag']*1e3:.3f} mm  eta_pred={cfg['eta_pred']:.4f}")

print()
for f in config.LIN_FREQUENCIES:
    t0, t1 = config.linearity_segment_window(f)
    print(f"  {f:4.0f} Hz  ->  [{t0:.1f}, {t1:.1f}] s")

---
## § 2  Raw signal overview (all 8 segments, ramps visible)

In [ ]:
for cfg in DATASETS:
    plotting.plot_raw_traces_uniform(
        cfg, components=('vx', 'vy', 'vz'),
        share_scale='dataset', max_rows=4, title=cfg['label'])

---
## § 3  Run the segment analysis

In [ ]:
for cfg in DATASETS:
    print(f"-- {cfg['label']} --")
    analysis.run_linearity_analysis(cfg, frequencies=config.LIN_FREQUENCIES)

---
## § 4  η(t) and η vs amplitude

Top: η(t), running-median smoothed; grey = fade region; dotted = η_pred.
Bottom: instantaneous |δL| (Hilbert envelope). Then η against |δL| — flat
binned medians = linear coupling; slope or hysteresis = amplitude
dependence.

In [ ]:
for cfg in DATASETS:
    plotting.plot_linearity_eta_time(cfg, cfg['lin_results'],
                                     smooth_cycles=5, ref='ends')

In [ ]:
plotting.plot_linearity_eta_vs_amplitude(
    DATASETS, smooth_cycles=5, exclude_fade=True, n_bins=25, ref='ends')

In [ ]:
for f_compare in [20, 100, 300]:
    plotting.plot_linearity_comparison(
        DATASETS, f_target=f_compare,
        smooth_cycles=5, exclude_fade=True, ref='ends', n_bins=25)

---
## § 5  Frequency-domain cross-check per segment

For each mono-frequency segment, the Welch H1 FRF between Δu_ends (input) and
δxₗ (output) evaluated at the segment frequency should match the time-domain
median η — a consistency check between the two estimation routes.

In [ ]:
from ldv_analysis import freqdomain

rows = []
for cfg in DATASETS:
    for f_t, res in cfg['lin_results'].items():
        fs = cfg['fs']
        f_w, H, coh = freqdomain.welch_transfer(
            res['delta_ends'], res['delta_xl'], fs,
            nperseg=int(fs))                     # 1 s segments -> df = 1 Hz
        k = int(np.argmin(np.abs(f_w - f_t)))
        rows.append(dict(dataset=cfg['label'], f_Hz=f_t,
                         eta_td=round(res['eta_med_ends'], 3),
                         eta_fd=round(float(np.abs(H[k])), 3),
                         coh=round(float(coh[k]), 3)))
pd.DataFrame(rows)

---
## § 6  Extension vs compression per amplitude bin

`asymmetry.linearity_segment_asymmetry` bins the ramp by instantaneous drive
amplitude and evaluates η separately over extension (δ_ref > 0) and
compression (δ_ref < 0) samples: diverging curves = amplitude-dependent
asymmetry (e.g. slack/slip developing on one half-cycle only).

In [ ]:
from ldv_analysis import asymmetry

for cfg in DATASETS:
    fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=True)
    for ax, f_t in zip(axes.ravel(), config.LIN_FREQUENCIES):
        d = asymmetry.linearity_segment_asymmetry(cfg, f_t, n_amp_bins=12)
        if d is None:
            ax.set_visible(False)
            continue
        ax.plot(d['amp_centers'] * 1e6, d['eta_ext'], 'o-', ms=3, color='C0',
                label='extension')
        ax.plot(d['amp_centers'] * 1e6, d['eta_comp'], 's-', ms=3, color='C3',
                label='compression')
        ax.axhline(cfg['eta_pred'], color='k', ls=':', lw=0.8)
        ax.set_title(f'{f_t:.0f} Hz', fontsize=9)
        ax.set_xlabel('|δL| [µm]')
    axes[0, 0].set_ylabel('η (half-cycle)')
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(f"{cfg['label']} — extension vs compression along the ramp")
    plt.tight_layout(); plt.show()

---
## § 7  Interactive dashboards

In [ ]:
from ldv_analysis import widgets
widgets.linearity_dashboard(DATASETS, frequencies=config.LIN_FREQUENCIES)

In [ ]:
widgets.linearity_slip_dashboard(DATASETS, frequencies=config.LIN_FREQUENCIES)

---
## § 8  Export

In [ ]:
acc = dict(labels=[], f=[], eta_med=[], eta_med_ends=[])
for cfg in DATASETS:
    for f_t, res in cfg['lin_results'].items():
        acc['labels'].append(cfg['label'])
        acc['f'].append(f_t)
        acc['eta_med'].append(res['eta_med'])
        acc['eta_med_ends'].append(res['eta_med_ends'])

export.export_results(
    'linearity_eta',
    meta=dict(frequencies=config.LIN_FREQUENCIES,
              segment_s=config.LIN_SEGMENT_DURATION,
              fade_s=config.LIN_FADE_DURATION),
    **acc)